# GNN Prediction Path Extraction And LLM Answer Demo

This notebook loads saved GNN answer-candidate predictions, reconstructs the matching WebQSP local graph from the processed cache, extracts all equal-length shortest paths, builds the deduplicated reasoning subgraph, and optionally sends it to an OpenAI chat model for the final answer.


In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pipeline").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {current}")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline import (
    ExtractShortestPathsStep,
    GenerateFinalAnswerStep,
    GnnPredictionCandidateScoringStep,
    Pipeline,
    StepContext,
)


In [2]:
evaluation_run_dir = repo_root / "data" / "webqsp" / "evaluations" / "1_20260519_224016"
predictions_path = evaluation_run_dir / "predictions.jsonl"
processed_instances_path = repo_root / "data" / "webqsp" / "processed" / "test_instances.pt"

prediction_index = 6  # change this from 0 to 9 for your 10 saved predictions
top_k = None  # use all saved GNN candidates for path extraction/LLM evidence
llm_model_id = "gpt-4.1-mini"
candidate_print_limit = 10
path_print_limit = 5
subgraph_print_limit = 80

print("Repo root:", repo_root)
print("Predictions file exists:", predictions_path.exists())
print("Processed instances file exists:", processed_instances_path.exists())


Repo root: /Users/andrea/Fax/NLPxSMIM/graphragx
Predictions file exists: True
Processed instances file exists: True


In [3]:
candidate_step = GnnPredictionCandidateScoringStep(
    predictions_path=predictions_path,
    processed_instances_path=processed_instances_path,
    prediction_index=prediction_index,
    top_k=top_k,
)
candidate_scores = candidate_step.execute(StepContext())
sample = candidate_scores.sample

print("Question:", sample.question)
print("Question entities:", sample.q_entities)
print("Gold answers:", sample.a_entities)
print("Graph triples:", len(sample.graph_triples))
print()
print(f"GNN candidates shown: first {candidate_print_limit} of {len(candidate_scores.candidates)}")
for rank, candidate in enumerate(candidate_scores.candidates[:candidate_print_limit], start=1):
    print(f"{rank}. {candidate.node_id}")
    print(f"   probability={candidate.probability:.6f} logit={candidate.logit:.6f}")
    print(f"   local_node_id={candidate.local_node_id} global_node_id={candidate.global_node_id}")
    print(f"   is_gold_answer={candidate.is_gold_answer} selection_reason={candidate.selection_reason}")

print()
print("Gold candidates selected by GNN:")
for candidate in candidate_scores.candidates:
    if candidate.is_gold_answer:
        print(f"- {candidate.node_id} (probability={candidate.probability:.6f}, rank={candidate_scores.candidates.index(candidate) + 1})")


Question: where to visit near bangkok
Question entities: ['Bangkok']
Gold answers: ['Bangkok International Trade and Exhibition Centre', 'Dusit Zoo', 'Wat Arun', 'Magic Land', 'Wat Ratchanatdaram', 'Erawan Shrine', 'Wat Pho', 'Khaosan Road', 'Wat Saket', 'Grand Palace', 'Wat Benchamabophit', 'Golden Buddha', 'Vimanmek Mansion', 'Samutprakarn Crocodile Farm and Zoo', 'Chatuchak Park', 'Ananta Samakhom Throne Hall', 'Democracy Monument', 'Thonburi', 'Rajamangala Stadium', 'Wat Suthat', 'Jim Thompson House', 'Bangkok Aquarium', 'Siam Park City', 'Safari World', 'Lumphini Park', 'Wat Phra Kaew', 'Bangkok National Museum']
Graph triples: 3942

GNN candidates shown: first 10 of 94
1. Bangkok
   probability=1.000000 logit=25.459581
   local_node_id=7 global_node_id=17094
   is_gold_answer=False selection_reason=threshold
2. The Hangover Part II
   probability=0.999989 logit=11.415400
   local_node_id=23 global_node_id=218998
   is_gold_answer=False selection_reason=threshold
3. Baraka
   prob

In [4]:
path_step = ExtractShortestPathsStep()
extracted_paths = path_step.execute(StepContext(result=candidate_scores))

print(f"Candidates with at least one path: {extracted_paths.found_paths}")
print(f"Candidates without a path: {extracted_paths.missing_paths}")
print(f"Deduplicated subgraph triples: {len(extracted_paths.reasoning_subgraph_triples)}")


Candidates with at least one path: 94
Candidates without a path: 0
Deduplicated subgraph triples: 484


In [5]:
for path in extracted_paths.paths[:path_print_limit]:
    print(f"Candidate: {path.candidate_node} (score={path.candidate_score:.4f})")

    if not path.path_found:
        print("  No path found.")
        print()
        continue

    print(f"  Equal-length shortest paths found: {len(path.shortest_paths)}")
    for index, shortest_path in enumerate(path.shortest_paths, start=1):
        print(f"  Path {index}:")
        for triple in shortest_path:
            print(f"    {triple.source} --[{triple.relation}]--> {triple.target}")
    print()

if len(extracted_paths.paths) > path_print_limit:
    print(f"... skipped {len(extracted_paths.paths) - path_print_limit} candidates in this display cell")


Candidate: Bangkok (score=1.0000)
  Equal-length shortest paths found: 1
  Path 1:

Candidate: The Hangover Part II (score=1.0000)
  Equal-length shortest paths found: 2
  Path 1:
    The Hangover Part II --[film.film.featured_film_locations]--> Bangkok
  Path 2:
    Bangkok --[film.film_location.featured_in_films]--> The Hangover Part II

Candidate: Baraka (score=0.9999)
  Equal-length shortest paths found: 2
  Path 1:
    Baraka --[film.film.featured_film_locations]--> Bangkok
  Path 2:
    Bangkok --[film.film_location.featured_in_films]--> Baraka

Candidate: The Great Challenge (score=0.9992)
  Equal-length shortest paths found: 2
  Path 1:
    The Great Challenge --[film.film.featured_film_locations]--> Bangkok
  Path 2:
    Bangkok --[film.film_location.featured_in_films]--> The Great Challenge

Candidate: Thailand (score=0.9980)
  Equal-length shortest paths found: 5
  Path 1:
    Bangkok --[base.biblioness.bibs_location.country]--> Thailand
  Path 2:
    Bangkok --[location.adm

In [6]:
print("Deduplicated reasoning subgraph sent to the LLM:")
print(f"Showing first {subgraph_print_limit} of {len(extracted_paths.reasoning_subgraph_triples)} triples")
print()

for triple in extracted_paths.reasoning_subgraph_triples[:subgraph_print_limit]:
    print(f"{triple.source} --[{triple.relation}]--> {triple.target}")

if len(extracted_paths.reasoning_subgraph_triples) > subgraph_print_limit:
    print(f"... skipped {len(extracted_paths.reasoning_subgraph_triples) - subgraph_print_limit} triples in this display cell")


Deduplicated reasoning subgraph sent to the LLM:
Showing first 80 of 484 triples

The Hangover Part II --[film.film.featured_film_locations]--> Bangkok
Bangkok --[film.film_location.featured_in_films]--> The Hangover Part II
Baraka --[film.film.featured_film_locations]--> Bangkok
Bangkok --[film.film_location.featured_in_films]--> Baraka
The Great Challenge --[film.film.featured_film_locations]--> Bangkok
Bangkok --[film.film_location.featured_in_films]--> The Great Challenge
Bangkok --[base.biblioness.bibs_location.country]--> Thailand
Bangkok --[location.administrative_division.country]--> Thailand
Thailand --[location.country.administrative_divisions]--> Bangkok
Thailand --[location.country.capital]--> Bangkok
Bangkok --[location.location.containedby]--> Thailand
Wanted --[film.film.featured_film_locations]--> Bangkok
Bangkok --[film.film_location.featured_in_films]--> Wanted
Baraka --[film.film.country]--> United States of America
Bangkok --[location.location.events]--> Bombing of 

In [7]:
pipeline = Pipeline(
    evaluation_steps=[
        GnnPredictionCandidateScoringStep(
            predictions_path=predictions_path,
            processed_instances_path=processed_instances_path,
            prediction_index=prediction_index,
            top_k=top_k,
        ),
        ExtractShortestPathsStep(),
        GenerateFinalAnswerStep(model_id=llm_model_id),
    ]
)

pipeline_result = pipeline.evaluate(StepContext())

print("Final answer:")
print(pipeline_result.final_result.answer)
print()
print("Prompt sent to LLM:")
print(pipeline_result.final_result.prompt)


/Users/andrea/miniconda3/envs/graphragx/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Final answer:
Wat Arun, Wat Benchamabophit, Wat Pho, Wat Phra Kaew, Wat Ratchanatdaram, Wat Saket, Wat Suthat, Chatuchak Park, Democracy Monument, Erawan Shrine, Golden Buddha, Khaosan Road, Lumphini Park, Samutprakarn Crocodile Farm and Zoo, Vimanmek Mansion, Grand Palace, Bangkok National Museum, Dusit Zoo, Rajamangala Stadium, Bangkok International Trade and Exhibition Centre, Safari World, Bangkok Aquarium, Thonburi

Prompt sent to LLM:
Question:
where to visit near bangkok

Reasoning paths:
Reasoning subgraph:
The Hangover Part II -> film.film.featured_film_locations -> Bangkok
Bangkok -> film.film_location.featured_in_films -> The Hangover Part II
Baraka -> film.film.featured_film_locations -> Bangkok
Bangkok -> film.film_location.featured_in_films -> Baraka
The Great Challenge -> film.film.featured_film_locations -> Bangkok
Bangkok -> film.film_location.featured_in_films -> The Great Challenge
Bangkok -> base.biblioness.bibs_location.country -> Thailand
Bangkok -> location.admin

#### Simple BFS Trace For One Candidate

This uses the `sample` reconstructed from the selected prediction. To keep the output readable, it only prints when BFS finds one of the equal-length shortest paths to the selected target.


In [16]:
from collections import deque
from pipeline.evaluation.services.shortest_path_extraction import ShortestPathExtractionService

service = ShortestPathExtractionService()
adjacency = service._build_undirected_adjacency(sample.graph_triples)

start_nodes = sorted(sample.q_entities)
target = candidate_scores.candidates[0].node_id  # change this to inspect another candidate

queue = deque((start, [], (start,)) for start in start_nodes)
shortest_paths = []
shortest_length = None
seen_path_keys = set()

print(f"Start: {', '.join(start_nodes)}")
print(f"Target: {target}")
print()

while queue:
    current, path, path_nodes = queue.popleft()

    if shortest_length is not None and len(path) >= shortest_length:
        continue

    for neighbor, triple in adjacency.get(current, []):
        if neighbor in path_nodes:
            continue

        new_path = path + [triple]

        if neighbor == target:
            shortest_length = len(new_path) if shortest_length is None else shortest_length
            path_key = tuple((t.source, t.relation, t.target) for t in new_path)

            if len(new_path) == shortest_length and path_key not in seen_path_keys:
                seen_path_keys.add(path_key)
                shortest_paths.append(new_path)
                print(f"Found shortest path {len(shortest_paths)} via {current} -> {neighbor}")
            continue

        if shortest_length is None or len(new_path) < shortest_length:
            queue.append((neighbor, new_path, (*path_nodes, neighbor)))

print()
print(f"Shortest path length: {shortest_length}")
print(f"Number of equal-length shortest paths: {len(shortest_paths)}")
print()

for index, shortest_path in enumerate(shortest_paths, start=1):
    print(f"Path {index}:")
    for triple in shortest_path:
        print(f"  {triple.source} --[{triple.relation}]--> {triple.target}")
    print()


Start: Nina Dobrev
Target: Nina Dobrev



KeyboardInterrupt: 